In [1]:
import os
import math
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
from tqdm import trange, tqdm
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import torch.nn.functional as F  # standard alias
# from a pythonfile import a class or function
from Utils.CADTensorGenerator import CADTensorGenerator
from Utils.CADVisualizer   import CADVisualizer
from HDVClassNet.PP_net import PPNet
from HDVClassNet.VoronoiDecorder import VoronoiDecoder

from Training.MainTrain import TrainingConfig, NN_Trainer
from neuraltomo_fem import run_fem_loss
from problems.TipCantilever_30_20_20_midLoad import TipCantilever_30_20_20_midLoad
from problems.ThickenShell import ThickenShell

import pyvista as pv
try:
    pv.set_jupyter_backend("trame")
except Exception:
    pass

# ---- Reproducibility (recommended for D_params comparisons) ----
SEED = 20
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BASE = Path(__file__).parent if "__file__" in globals() else Path.cwd()
print("Code Directory:", BASE)
TesPartsDir = BASE / "Testparts" 
print("Test Step files Directory:", TesPartsDir)
# ---- Device ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


Code Directory: /home/arash/VoronoiImp-main
Test Step files Directory: /home/arash/VoronoiImp-main/Testparts
device: cuda


In [2]:
viz = CADVisualizer()
# Laoding model and extracting mesh and tensors as input
FreeFormSurf1  = TesPartsDir / "FreeFormCrv1.stp"
FreeFormSurf2A = TesPartsDir / "FreeFormSurf2A.STEP"
YachtBodypart  = TesPartsDir / "YachtBodypart.stp"
CircularSurf1  = TesPartsDir / "CircularSurf1.stp"
Cube           = TesPartsDir / "Cube.stp"
CircularSur2   = TesPartsDir / "CircularSur2.stp"
Conic          = TesPartsDir / "Conic.stp"
CircularHoles  = TesPartsDir / "CircularHoles.stp"
FullCylinder   = TesPartsDir / "FullCylinder.stp"
Sphere         = TesPartsDir / "Sphere.stp"
SphereTap      = TesPartsDir / "SphereTap.stp"
Tidebottle     = TesPartsDir / "Tidebottle.STEP"

shape_path = str(FullCylinder)
generator = CADTensorGenerator(
    deflection=0.001,
    angle=0.5,
    metric_tol=1e-9,
    det_min=1e-5,
    n_u=100,
    n_v=100,
    device=device,
)
mesh_df, faces_df, tensors = generator.generate_from_file(shape_path, input_ring=2)

# mesh_df, faces_df, tensors = generator.generate_from_file(
#     shape_path=shape_path,
#     input_ring=1,
#     visualize=True,
# )
print(tensors["num_faces"])
print(tensors["face_tensors"][0].keys())
print(tensors["face_periodicity"])
uv = tensors["uv"]
points_xyz = tensors["points_xyz"]
uv = tensors["uv"]
Xu = tensors["Xu"]
Xv = tensors["Xv"]
points_xyz = tensors["points_xyz"]
face_areas = tensors["face_areas"]
faces_ijk = tensors["faces_ijk"]
face_id = tensors["face_id"]
boundary_idx_ring1 = tensors["boundary_idx_ring1"]
pv_faces = tensors["pv_faces"]
viz.visualize_show_Model(points_xyz, pv_faces)
tensors["BBX"]
print("BBX:", tensors["BBX"])

MinVolFrac: 0.05000000074505806
uv device: cuda:0
1
dict_keys(['face_id', 'uv', 'points_xyz', 'Xu', 'Xv', 'faces_ijk', 'pv_faces', 'face_areas', 'boundary_idx', 'boundary_idx_ring1', 'boundary_idx_ring2', 'min_vol_frac', 'BBX', 'u_periodic', 'v_periodic', 'u_period', 'v_period', 'u_raw_bounds', 'v_raw_bounds', 'global_vertex_idx', 'global_face_idx', 'num_vertices', 'num_faces'])
{0: {'u_periodic': True, 'v_periodic': False, 'u_period': 6.283185307179586, 'v_period': None}}


Widget(value='<iframe src="http://localhost:35349/index.html?ui=P_0x73bc4b7aa620_0&reconnect=auto" class="pyvi…

BBX: {0: {'xmin': -41.0, 'xmax': 41.0, 'ymin': -41.0, 'ymax': 41.0, 'zmin': 0.0, 'zmax': 100.0}}


In [ ]:
fixed_height_shell= 1.0 
shell_problem = ThickenShell(
    thickness=fixed_height_shell,
    BC_dir = "z",
    Load_magnitude=1.0,
    voxel_size=1.0,
    extra_layers=0,
    tensors=tensors,
    tangential_tol=0.5,
)

fem = run_fem_loss.NeuralTOMOFEM(shell_problem, device=device, isotropic=False)
# shell_problem.debug_voxel_stats()
shell_problem.show_voxels_surface_and_bc_NEW()

In [ ]:
cfg = TrainingConfig(
    seed_number=10,
    use_anisotropy=False,
    fixed_height=fixed_height_shell,
    target_volfrac=0.5,
    seed_repulsion_sigma=0.08,
    boundary_margin=0.05,

    lam_fem=2.0,
    lam_vol = 1.0,
    lam_rep = 0.2,
    lam_bnd = 0.01,

    lam_best_vol=5.0,
    lam_best_fem=0.0,

    comp_normalize_by=1e10,
    normalize_losses=False,

    fem_density_floor=0.02,
    skip_bad_fem_steps=True,

    num_steps=10000,
    context_vector_size=8,
    tau=0.01,
    beta=0.05,
    log_every=50,
    early_stop_start=200,
    patience=200,
    min_delta=1e-8,
)
trainer = NN_Trainer(
    generator=generator,
    viz=viz,
    decoder_cls=VoronoiDecoder,
    ppnet_cls=PPNet,
    fem=fem,
    shell_problem=shell_problem,
    config=cfg,
)

result = trainer.train(
    uv=uv,
    Xu=Xu,
    Xv=Xv,
    points_xyz=points_xyz,
    face_areas=face_areas,
    faces_ijk=faces_ijk,
    face_id=face_id,
    boundary_idx_ring1=boundary_idx_ring1,
    face_tensors=tensors["face_tensors"],   # <- add this
)

trainer.visualize_result_stepwise(result, points_xyz, faces_ijk)
trainer.visualize_result_final(result, points_xyz, faces_ijk, thr=0.01, show_solid=True)